# Phase 1 — Exploratory Data Analysis

IBM Telco Customer Churn: understand data quality, target balance, feature relationships, and leakage risks before modeling.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.load import load_raw_data, load_config
from src.data.validate import validate_dataframe, prepare_target

sns.set_theme(style="whitegrid", context="notebook")
cfg = load_config()
raw = load_raw_data()
report = validate_dataframe(raw)
df = prepare_target(raw)

print(f"Rows: {report.n_rows}, Cols: {report.n_cols}")
print("Target distribution:", report.target_distribution)
print("Issues:", report.issues or "none")
df.head()

## Data validation summary

- Expected schema matches the IBM Telco sample (21 columns).
- `TotalCharges` is often stored as a string; blank values typically correspond to brand-new customers (`tenure == 0`).
- Duplicate rows / customer IDs should be near zero for this public sample.

In [ ]:
print("Dtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())
print("\nTotalCharges missing with tenure:")
print(df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]].head(10))

### Observation — missing TotalCharges

Blank `TotalCharges` align with `tenure == 0`. That is a data-entry artifact, not random missingness. Impute with 0 (or `MonthlyCharges`) before modeling; do not drop these rows without reason.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df["Churn"].value_counts().rename({0: "No", 1: "Yes"}).plot(
    kind="bar", ax=ax, color=["#4C78A8", "#E45756"]
)
ax.set_title("Target distribution (Churn)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

churn_rate = df["Churn"].mean()
print(f"Churn rate: {churn_rate:.1%}")
print("Implication: accuracy alone is misleading — a majority-class model gets ~73% accuracy while recalling 0% of churners.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(df, x=col, hue="Churn", bins=30, ax=ax, element="step", stat="density", common_norm=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### Observation — tenure and charges

- Churners concentrate at **low tenure** (early-life customers).
- Higher **MonthlyCharges** associate with higher churn density.
- `TotalCharges` correlates with tenure × monthly bill; useful but partially redundant with those two features.

In [ ]:
cat_cols = ["Contract", "InternetService", "PaymentMethod", "Partner", "Dependents", "PaperlessBilling"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    rates = df.groupby(col)["Churn"].mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="#E45756")
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("Churn rate")
    ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.show()

### Observation — categorical relationships

- **Month-to-month** contracts churn far more than one-/two-year contracts (strong signal).
- Fiber optic internet and electronic check payment methods show elevated churn.
- Modeling implication: categorical encoding of contract/payment/internet service is essential; linear models can capture much of this signal.

## Leakage investigation

Questions to ask: does any feature encode the churn outcome after the fact?

| Feature | Leakage risk | Notes |
|---|---|---|
| `Churn` | Target | Must not be used as a feature |
| `customerID` | Identifier | Drop from modeling |
| `TotalCharges` | Low | Correlated with tenure; not post-outcome |
| Service flags | Low | Snapshot of current plan |
| Contract / tenure | Low–medium | Reflect current state; OK for this static dataset |

This dataset is a **cross-sectional snapshot**, not a true 30-day forward-looking label. We treat `Churn == Yes` as the binary label for educational purposes. In production you would label "churned within next 30 days" from event timestamps and carefully exclude post-cutoff features.

In [ ]:
# Correlation among numeric features + target
num = df[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].corr()
sns.heatmap(num, annot=True, cmap="RdBu_r", center=0)
plt.title("Numeric correlations")
plt.tight_layout()
plt.show()

print("tenure vs TotalCharges correlation is high — expect multicollinearity; LR may down-weight one of them.")

## Modeling implications (Phase 1 takeaways)

1. **Class imbalance (~27% churn)** → report precision/recall/F1/PR-AUC, not just accuracy.
2. **Strong categorical drivers** (Contract, InternetService, PaymentMethod) → one-hot encoding in the pipeline.
3. **Impute TotalCharges** for tenure-0 customers.
4. **Drop customerID**; keep features available at scoring time only.
5. **Baselines next**: majority class and a simple contract-based rule to define "better than trivial."